# Módulo 4: Introducción al Machine Learning

Este notebook de nivel de módulo proporciona un resumen rápido de las diez lecciones con ejemplos ejecutables.

## Objetivos de Aprendizaje

- Comprender el workflow de ML desde los datos hasta el despliegue
- Entrenar modelos supervisados: regresión, clasificación, árboles de decisión, random forest, gradient boosting
- Entrenar modelos no supervisados: K-Means, PCA
- Interpretar outputs de modelos
- Aplicar ML a problemas de biotecnología y SaaS

## Prerrequisitos

- Módulo 3: Estadística para Machine Learning
- Python: numpy, pandas, matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print("All imports successful.")

All imports successful.


## L1: Fundamentos de ML

Features, labels, training, predicción, división train/test.

In [2]:
from sklearn.datasets import load_diabetes
data = load_diabetes()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
print(f"L1: Train R² = {lr.score(X_train, y_train):.3f}, Test R² = {lr.score(X_test, y_test):.3f}")

L1: Train R² = 0.528, Test R² = 0.453


## L2: Regresión Lineal

OLS, coeficientes, MSE, RMSE, R².

In [3]:
y_pred = lr.predict(X_test)
print(f"L2: MSE = {mean_squared_error(y_test, y_pred):.1f}")
print(f"L2: RMSE = {np.sqrt(mean_squared_error(y_test, y_pred)):.1f}")
print(f"L2: R² = {r2_score(y_test, y_pred):.3f}")
print(f"Top coefficients: bmi={lr.coef_[2]:.1f}, s5={lr.coef_[8]:.1f}")

L2: MSE = 2900.2
L2: RMSE = 53.9
L2: R² = 0.453
Top coefficients: bmi=542.4, s5=736.2


## L3: Clasificación

Regresión logística, matriz de confusión, ROC AUC.

In [4]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_tr, y_tr)
y_pred = logreg.predict(X_te)
print(f"L3: Accuracy = {accuracy_score(y_te, y_pred):.3f}")
print(confusion_matrix(y_te, y_pred))

L3: Accuracy = 0.956
[[39  4]
 [ 1 70]]


## L4: Árboles de Decisión

Particionamiento recursivo, impureza de Gini, pruning.

In [5]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_tr, y_tr)
print(f"L4: Tree accuracy = {tree.score(X_te, y_te):.3f}")
print(f"Top feature: {bc.feature_names[tree.feature_importances_.argmax()]}")

L4: Tree accuracy = 0.947
Top feature: mean concave points


## L5: Random Forest

Ensemble learning, bagging, importancia de features.

In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"L5: RF accuracy = {rf.score(X_te, y_te):.3f}")
imp = pd.DataFrame({'feature': bc.feature_names, 'importance': rf.feature_importances_})
print(imp.sort_values('importance', ascending=False).head(3).to_string(index=False))

L5: RF accuracy = 0.965
             feature  importance
          worst area    0.153892
worst concave points    0.144663
 mean concave points    0.106210


## L6: K-Means Clustering

Unsupervised learning, método del codo, silhouette score.

In [7]:
from sklearn.datasets import make_blobs
X_blob, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_blob)
print(f"L6: Inertia = {kmeans.inertia_:.1f}")
print(f"L6: Cluster centers:\n{kmeans.cluster_centers_}")

L6: Inertia = 362.5
L6: Cluster centers:
[[-2.63715917  8.98563949]
 [-6.84180708 -6.84038791]
 [ 4.70253968  2.02807134]
 [-8.83330596  7.21790214]]


## L7: PCA

Reducción de dimensionalidad, varianza explicada.

In [8]:
X_scaled = StandardScaler().fit_transform(bc.data)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"L7: 2 components explain {pca.explained_variance_ratio_.sum():.1%} of variance")
print(f"L7: Per component: {pca.explained_variance_ratio_}")

L7: 2 components explain 63.2% of variance
L7: Per component: [0.44272026 0.18971182]


## L8: Gradient Boosting

Ensemble secuencial, learning rate, early stopping.

In [9]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
print(f"L8: GB accuracy = {gb.score(X_te, y_te):.3f}")

L8: GB accuracy = 0.956


## L9: Interpretación de Modelos

Importancia por permutación, gráficos de dependencia parcial.

In [10]:
from sklearn.inspection import permutation_importance
result = permutation_importance(rf, X_te, y_te, n_repeats=5, random_state=42)
top_idx = result.importances_mean.argsort()[::-1][:3]
print("L9: Top 3 permutation importances:")
for i in top_idx:
    print(f"  {bc.feature_names[i]}: {result.importances_mean[i]:.4f}")

L9: Top 3 permutation importances:
  worst texture: 0.0035
  worst radius: 0.0035
  mean texture: 0.0035


## L10: Aplicaciones

Pipelines completos con tipos de datos mixtos.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

n = 500
bio = pd.DataFrame({
    'temperature': np.random.normal(37, 2, n),
    'ph': np.random.normal(7.2, 0.3, n),
    'batch_type': np.random.choice(['A', 'B', 'C'], n),
})
bio['quality'] = (80 - 2*np.abs(bio['temperature']-37) - 5*np.abs(bio['ph']-7.2) + np.random.normal(0,3,n)).clip(0,100)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['temperature', 'ph']),
    ('cat', OneHotEncoder(drop='first'), ['batch_type']),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42)),
])

X_bio, y_bio = bio.drop('quality', axis=1), bio['quality']
grid = GridSearchCV(pipeline, {'model__n_estimators': [50, 100]}, cv=3, scoring='r2')
grid.fit(X_bio, y_bio)
print(f"L10: Best CV R² = {grid.best_score_:.3f}")
print(f"L10: Best params = {grid.best_params_}")

L10: Best CV R² = 0.279
L10: Best params = {'model__n_estimators': 100}


## Resumen

El Módulo 4 cubrió el workflow completo de Machine Learning:

| Lección | Algoritmo | Tipo |
|---------|-----------|------|
| L1 | Fundamentos de ML | Conceptos |
| L2 | Regresión Lineal | Supervisado (Regresión) |
| L3 | Regresión Logística | Supervisado (Clasificación) |
| L4 | Árboles de Decisión | Supervisado |
| L5 | Random Forest | Supervisado (Ensemble) |
| L6 | K-Means | No supervisado |
| L7 | PCA | No supervisado |
| L8 | Gradient Boosting | Supervisado (Ensemble) |
| L9 | Interpretación de Modelos | Explicabilidad |
| L10 | Aplicaciones | Extremo a extremo |

¡Siguiente: Proyecto Final!